# Lets do cosior analysis on the data expressions!

In [44]:
from circadian_cell_painting import (
    detect,
    CosinorDetection,
    CosinorVisualize,
    OscillationResults
)
from pathlib import Path
import polars as pl
import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path
import string
from tqdm import tqdm
import numpy as np
from scipy.signal import savgol_filter
import pandas as pd
from natsort import natsorted
import matplotlib.colors as mcolors

dataset = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/"
               "enVision/enVision_preprocessed/2026-04-01._.PREPROCESSED."
               "parquet")


lf = pl.scan_parquet(dataset).filter(pl.col("feature") == "CalcResultI")

tpoints = lf.select("Timepoint").unique().sort(by="Timepoint").collect().to_series().to_list()

platemap = (
    pl.read_excel(
    "/home/derfelt/PartchLabFiles/Circadian"
    " Rhythms/Live Study/2026-04-01Echo_transfer.xlsx", engine="openpyxl")
    .select(["384_Well", 
             "MoleculeID", 
             "strain",
             "dex"])
    .unique(subset=["384_Well"])
    .with_columns(
        pl.col("MoleculeID").str.strip_chars()
    )
    ).rename({"384_Well": "Well"})

display(lf.head(2).collect())
display(platemap)

# print(*tpoints,sep="\n")

Well,Timepoint,feature,mean,median,stddev,q0,q10,q25,q50,q75,q90,q100
str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""A01""",24.0,"""CalcResultI""",589.67097,589.67097,null,589.67097,589.67097,589.67097,589.67097,589.67097,589.67097,589.67097
"""A01""",24.5,"""CalcResultI""",425.256143,425.256143,null,425.256143,425.256143,425.256143,425.256143,425.256143,425.256143,425.256143


Well,MoleculeID,strain,dex
str,str,str,str
"""J11""","""0.05% DMSO""","""Per2""","""dex"""
"""G14""","""0.01% DMSO""","""Bmal1""","""dex"""
"""I06""","""0.01% DMSO""","""Per2""","""dex"""
"""F10""","""noDMSO""","""Bmal1""","""NoSynhronisation"""
"""I12""","""Etoposide 2uM""","""Per2""","""dex"""
…,…,…,…
"""M18""","""noDMSO""","""Per2""","""NoSynhronisation"""
"""E18""","""Aswad A0024""","""Bmal1""","""dex"""
"""K10""","""KL001 4uM""","""Per2""","""dex"""


In [4]:
out_file = Path("/home/derfelt/PartchLabFiles/Circadian "
                "Rhythms/Live Study/enVision/cosinor_anal"
                "ysis/2026-04._.RESULTS.parquet")

model = CosinorDetection(intervals=0.5)

result: OscillationResults = detect(
    file_path=dataset,
    platemap_path=platemap,
    feature_col="feature",
    well_column="Well",
    timepoint_col="Timepoint",
    value_col="mean",
    stat="mean",
    out_file=out_file,
    correction_group=["MoleculeID", "strain"],
    detector=model
)



Well progress:   0%|          | 0/384 [00:00<?, ?it/s]

Well progress: 100%|██████████| 384/384 [00:05<00:00, 67.15it/s]


In [5]:
print(result)

OscillationResults(well_col='Well', feature_col='feature', timepoint_col='Timepoint', value_col='mean', stat='mean', detector_params={'period': 24, 'time_interval': 0.5, 'time_offset': 0, 'start_index': 0}, results_path=PosixPath('/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/cosinor_analysis/2026-04._.RESULTS.parquet'), data_path=PosixPath('/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/enVision_preprocessed/2026-04-01._.PREPROCESSED.parquet'), platemap=None, detector_name='CosinorFitting', adj_alpha=0.05, adj_method='fdr_bh', correction_group=['MoleculeID', 'strain'])


# let's visualize the results

In [56]:
result_lf = pl.scan_parquet(result.results_path).filter(pl.col("feature") == "CalcResultI")
dataset_lf = pl.scan_parquet(dataset).filter(pl.col("feature") == "CalcResultI").select(["Well", "Timepoint", "feature", "mean"])
dataset_lf = dataset_lf.join(other=platemap.lazy(), on="Well", how="left").sort(["Well", "Timepoint"])

# smoothing option
dataset_lf = (dataset_lf
                .collect()
                .group_by("Well")
                .map_groups(
                    lambda g: g.with_columns(
                      pl.Series("mean", savgol_filter(
                          g["mean"].to_numpy(),
                          window_length=13,
                          polyorder=3
                      ))
                    )
                )
              )

#grab only dex
dataset_lf = (dataset_lf
              .filter(pl.col('dex') == 'dex')
              .sort(["dex","strain","MoleculeID", "Well", "Timepoint"], descending=[True, True, True, False, False]))

display(dataset_lf)

# display(dataset_lf.head(10).collect())
# display(result_lf.head(10).collect().select())

# print(result_lf.collect().shape)

Well,Timepoint,feature,mean,MoleculeID,strain,dex
str,f64,str,f64,str,str,str
"""P01""",24.0,"""CalcResultI""",-1012.525894,"""food_coloring""",null,"""dex"""
"""P01""",24.5,"""CalcResultI""",-891.890558,"""food_coloring""",null,"""dex"""
"""P01""",25.0,"""CalcResultI""",-777.546248,"""food_coloring""",null,"""dex"""
"""P01""",25.5,"""CalcResultI""",-668.600662,"""food_coloring""",null,"""dex"""
"""P01""",26.0,"""CalcResultI""",-564.161501,"""food_coloring""",null,"""dex"""
…,…,…,…,…,…,…
"""C13""",91.5,"""CalcResultI""",-215.166266,"""0.005% DMSO""","""Bmal1""","""dex"""
"""C13""",92.0,"""CalcResultI""",-221.396754,"""0.005% DMSO""","""Bmal1""","""dex"""
"""C13""",92.5,"""CalcResultI""",-231.291174,"""0.005% DMSO""","""Bmal1""","""dex"""


In [ ]:
# collect and join
import pandas as pd
from scipy.signal import savgol_filter

dataset_pddf = dataset_lf.to_pandas()
wells_needed = dataset_pddf["Well"].unique()
result_pddf = result_lf.collect().to_pandas()

timepoints = sorted(dataset_pddf["Timepoint"].unique())

# display(dataset_pddf, result_pddf)


def expand_fitted(result_pddf, tpoints):
    rows = []
    for _, row in result_pddf.iterrows():
        fitted_vals = row["fitted"]          # list of floats
        if fitted_vals is None or len(fitted_vals) != len(tpoints):
            continue
        for t, f in zip(tpoints, fitted_vals):
            rows.append({"Well": row["Well"], "Timepoint": t, "fitted": f,
                         "MoleculeID": row["MoleculeID"], "strain": row["strain"],
                         "dex": row["dex"], "baseline": row["baseline"],
                         "amplitude": row["amplitude"], "phase": row["phase"],
                         "p_value": row["p_value"], "p_adjusted": row["p_adjusted"]})
    return pd.DataFrame(rows)

fitted_long = expand_fitted(result_pddf, timepoints)

display(fitted_long, dataset_pddf)

,Well,Timepoint,fitted,MoleculeID,strain,dex,baseline,amplitude,phase,p_value,p_adjusted
0,A01,24.0,36.886032,NaN,NaN,NaN,3.862568,171.55320,1.764502,8.329984e-07,1.643483e-06
1,A01,24.5,25.805053,NaN,NaN,NaN,3.862568,171.55320,1.764502,8.329984e-07,1.643483e-06
2,A01,25.0,14.630114,NaN,NaN,NaN,3.862568,171.55320,1.764502,8.329984e-07,1.643483e-06
3,A01,25.5,3.409066,NaN,NaN,NaN,3.862568,171.55320,1.764502,8.329984e-07,1.643483e-06
4,A01,26.0,-7.810040,NaN,NaN,NaN,3.862568,171.55320,1.764502,8.329984e-07,1.643483e-06
...,...,...,...,...,...,...,...,...,...,...,...
53371,P24,91.5,133.546002,NaN,NaN,NaN,-198.592260,1451.23053,4.354254,4.923728e-12,3.267565e-11
53372,P24,92.0,40.439105,NaN,NaN,NaN,-198.592260,1451.23053,4.354254,4.923728e-12,3.267565e-11
53373,P24,92.5,-53.691360,NaN,NaN,NaN,-198.592260,1451.23053,4.354254,4.923728e-12,3.267565e-11
53374,P24,93.0,-148.442314,NaN,NaN,NaN,-198.592260,1451.23053,4.354254,4.923728e-12,3.267565e-11


,Well,Timepoint,feature,mean,MoleculeID,strain,dex
0,P01,24.0,CalcResultI,-1012.525894,food_coloring,NaN,dex
1,P01,24.5,CalcResultI,-891.890558,food_coloring,NaN,dex
2,P01,25.0,CalcResultI,-777.546248,food_coloring,NaN,dex
3,P01,25.5,CalcResultI,-668.600662,food_coloring,NaN,dex
4,P01,26.0,CalcResultI,-564.161501,food_coloring,NaN,dex
...,...,...,...,...,...,...,...
27239,C13,91.5,CalcResultI,-215.166266,0.005% DMSO,Bmal1,dex
27240,C13,92.0,CalcResultI,-221.396754,0.005% DMSO,Bmal1,dex
27241,C13,92.5,CalcResultI,-231.291174,0.005% DMSO,Bmal1,dex
27242,C13,93.0,CalcResultI,-245.099249,0.005% DMSO,Bmal1,dex


In [24]:
PLOT_CONFIGS = [
    {
        'title': 'DMSO Controls',
        'drugs': [],
        'dmso' : ['0.005% DMSO', '0.01% DMSO', '0.05% DMSO',
                  '0.1% DMSO', '0.25% DMSO', '0.5% DMSO', '1% DMSO'],
    },
    {
        'title': 'MH-11 & MH-23',
        'drugs': [('MH-11', 'steelblue'), ('MH-23', 'darkorange')],
        'dmso' : ['0.05% DMSO'],
    },
    {
        'title': 'MH-13 & MH-20',
        'drugs': [('MH-13', 'steelblue'), ('MH-20', 'darkorange')],
        'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
    },
    {
        'title': 'PF & Etoposide',
        'drugs': [('PF', 'steelblue'), ('Etoposide', 'darkorange')],
        'dmso' : ['0.01% DMSO'],
    },
    {
        'title': 'LY, SR, Nobiletin, Naringine, T0901317',
        'drugs': [
            ('LY', 'steelblue'), ('SR', 'darkorange'),
            ('Nob', 'green'), ('Nar', 'purple'), ('T0901317', 'brown')
        ],
        'dmso' : ['0.05% DMSO', '0.01% DMSO'],
    },
    {
        'title': 'Bacterial & Aswad',
        'drugs': [
            ('E.coli#36', 'steelblue'), ('S.pneumo#6', 'darkorange'),
            ('Aswad A0005', 'green'), ('Aswad A0010', 'purple'), ('Aswad A0024', 'brown')
        ],
        'dmso' : ['0.05% DMSO'],
    },
    {
        'title': 'KL001',
        'drugs': [('KL001', 'steelblue')],
        'dmso' : ['0.01% DMSO'],
    },
    {
        'title': 'MK Compounds',
        'drugs': [
            ('MK-0761', 'steelblue'), ('MK-0769', 'darkorange'),
            ('MK-1282', 'green'),     ('MK-1496', 'purple'),
            ('MK-1804', 'brown'),     ('MK-2267', 'red'),
            ('MK-8692', 'teal'),      ('MK-8800', 'olive'),
        ],
        'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
    },
    {
        'title' : 'MK-0761',
        'drugs' : [('MK-0761', 'steelblue')],
        'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
    },
    {
        'title' : 'MK-1282',
        'drugs' : [('MK-1282', 'green')],
        'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
    },
    {
        'title' : 'MK-1804',
        'drugs' : [('MK-1804', 'brown')],
        'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
    },
    {
        'title' : 'MK-8692',
        'drugs' : [('MK-8692', 'teal')],
        'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
    },
    {
        'title' : 'MK-0769',
        'drugs' : [('MK-0769', 'darkorange')],
        'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
    },
    {
        'title' : 'MK-1496',
        'drugs' : [('MK-1496', 'purple')],
        'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
    },
    {
        'title' : 'MK-2267',
        'drugs' : [('MK-2267', 'red')],
        'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
    },
    {
        'title' : 'MK-8800',
        'drugs' : [('MK-8800', 'olive')],
        'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
    },
]

In [ ]:
def get_shades(base_color: str, n: int) -> list:
    """light → dark shades of base_color"""
    rgb = mcolors.to_rgb(base_color)
    return [tuple(1 - (1 - c) * (i + 1) / n for c in rgb) for i in range(n)]

def get_grays(n: int) -> list:
    return [str(round(0.3 + 0.6 * i / max(n - 1, 1), 2)) for i in range(n)]

def add_period_marks(ax, x_min, x_max, interval=12):
    marks = np.arange(np.ceil(x_min / interval) * interval, x_max + interval, interval)
    for mark in marks:
        ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
    ax.set_xticks(marks)
    ax.set_xticklabels([f'{int(m)}h' for m in marks], fontsize=7)

def plot_config_cosinor(ax, dataset_df, fitted_df, config, strain=None, colored_dmso=True):
    """
    Plots raw detrended mean (with SD shading) + cosinor fitted curve
    for each treatment group.
    """
    d_df = dataset_df.copy()
    f_df = fitted_df.copy()

    if strain:
        d_df = d_df[d_df['strain'] == strain]
        f_df = f_df[f_df['strain'] == strain]

    dmso_list = config['dmso']

    def _plot_pair(sub_raw, sub_fit, color, label, ax, linewidth=1.5, zorder=5):
        """Plot raw + fitted for a single group"""
        if sub_raw.empty:
            return
        # raw signal with SD shading
        sns.lineplot(
            data=sub_raw, x='Timepoint', y='mean',
            errorbar='sd', color=color, linewidth=linewidth,
            label=label, ax=ax, zorder=zorder,
            err_kws={'alpha': 0.15, 'facecolor': color}
        )
        # fitted curve — mean across replicates, dashed
        if not sub_fit.empty:
            fit_mean = sub_fit.groupby('Timepoint')['fitted'].mean().reset_index()
            ax.plot(
                fit_mean['Timepoint'], fit_mean['fitted'],
                color=color, linewidth=linewidth,
                linestyle='--', zorder=zorder, alpha=0.8
            )

    # ── DMSO controls ─────────────────────────────────────────────────────────
    if not config['drugs'] and colored_dmso:
        palette = sns.color_palette("tab10", len(dmso_list))
        for dmso, color in zip(dmso_list, palette):
            _plot_pair(
                d_df[d_df['MoleculeID'] == dmso],
                f_df[f_df['MoleculeID'] == dmso],
                color=color, label=dmso, ax=ax, linewidth=2, zorder=10
            )
    else:
        # ── Drug groups ───────────────────────────────────────────────────────
        for drug, base_color in config['drugs']:
            drug_raw = d_df[d_df['MoleculeID'].str.startswith(drug, na=False)]
            drug_fit = f_df[f_df['MoleculeID'].str.startswith(drug, na=False)]
            concentrations = natsorted(drug_raw['MoleculeID'].unique().tolist())
            shades = get_shades(base_color, len(concentrations))

            for conc, shade in zip(concentrations, shades):
                _plot_pair(
                    drug_raw[drug_raw['MoleculeID'] == conc],
                    drug_fit[drug_fit['MoleculeID'] == conc],
                    color=shade, label=conc, ax=ax
                )

        # ── DMSO on top in black ──────────────────────────────────────────────
        grays = get_grays(len(dmso_list))
        for dmso, gray in zip(dmso_list, grays):
            _plot_pair(
                d_df[d_df['MoleculeID'] == dmso],
                f_df[f_df['MoleculeID'] == dmso],
                color='black', label=dmso, ax=ax, linewidth=2, zorder=10
            )

    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
    title = config['title']
    if strain:
        title += f' — {strain}'
    ax.set_title(title)
    ax.set_xlabel('Time (hours)')
    ax.set_ylabel('Expression')
    ax.legend(fontsize=6, loc='upper right', ncol=2)
    add_period_marks(ax, d_df['Timepoint'].min(), d_df['Timepoint'].max())

timepoint_col = "Timepoint"
value = "mean"
meta_cols = ['MoleculeID', 'strain', 'dex']

strains = dataset_pddf['strain'].dropna().unique()
out_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/enVision_visual_results/cosinor_fitted.pdf")
with PdfPages(out_path) as pdf:
    for config in tqdm(PLOT_CONFIGS, desc="Plot Configs"):

        fig, axs = plt.subplots(2, 1, figsize=(24, 10), sharey=False)
        for ax, strain in zip(axs, strains):
            plot_config(ax=ax, df=dataset_pddf, fit_df=fitted_long,
                        config=config, strain=strain, colored_dmso=True)

        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

Plot Configs: 100%|██████████| 16/16 [00:10<00:00,  1.60it/s]
